# Retail Sales Data Preprocessing

## Project Overview
This notebook performs comprehensive data preprocessing on a retail sales dataset. We'll handle data cleaning, validation, feature engineering, and exploratory analysis to prepare the data for downstream analysis and modeling.

**Dataset**: Retail sales transactions with customer, product, and transaction details.

## Step 1: Import Necessary Libraries

We'll import essential libraries for data manipulation, analysis, and visualization.

In [1]:
# Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options for better visibility
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ All libraries imported successfully!")

✓ All libraries imported successfully!


## Step 2: Load the Dataset

Load the retail sales data from the CSV file and perform initial inspection.

In [2]:
# Load the dataset from CSV file
df = pd.read_csv('dataset/retail_sales_dataset.csv')

# Display success message and basic info
print(f"✓ Dataset loaded successfully!")
print(f"File shape: {df.shape[0]} rows, {df.shape[1]} columns")

✓ Dataset loaded successfully!
File shape: 1000 rows, 9 columns


## Step 3: Initial Data Exploration

Examine the structure and content of the dataset to understand what we're working with.

In [3]:
# Display first 5 rows
print("=" * 80)
print("FIRST 5 ROWS OF THE DATASET")
print("=" * 80)
print(df.head())

# Display dataset shape
print("\n" + "=" * 80)
print("DATASET SHAPE")
print("=" * 80)
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

# Display column names
print("\n" + "=" * 80)
print("COLUMN NAMES")
print("=" * 80)
print(df.columns.tolist())

# Display data types
print("\n" + "=" * 80)
print("DATA TYPES")
print("=" * 80)
print(df.dtypes)

# Display summary statistics
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)
print(df.describe())

FIRST 5 ROWS OF THE DATASET
   Transaction ID        Date Customer ID  Gender  Age Product Category  \
0               1  2023-11-24     CUST001    Male   34           Beauty   
1               2  2023-02-27     CUST002  Female   26         Clothing   
2               3  2023-01-13     CUST003    Male   50      Electronics   
3               4  2023-05-21     CUST004    Male   37         Clothing   
4               5  2023-05-06     CUST005    Male   30           Beauty   

   Quantity  Price per Unit  Total Amount  
0         3              50           150  
1         2             500          1000  
2         1              30            30  
3         1             500           500  
4         2              50           100  

DATASET SHAPE
Rows: 1000
Columns: 9

COLUMN NAMES
['Transaction ID', 'Date', 'Customer ID', 'Gender', 'Age', 'Product Category', 'Quantity', 'Price per Unit', 'Total Amount']

DATA TYPES
Transaction ID      int64
Date                  str
Customer ID      

## Step 4: Data Cleaning

Clean the dataset by handling missing values, duplicates, and converting data types appropriately.

In [4]:
# Check for missing values
print("=" * 80)
print("MISSING VALUES CHECK")
print("=" * 80)
missing_values = df.isnull().sum()
if missing_values.sum() > 0:
    print("Missing values found:")
    print(missing_values[missing_values > 0])
else:
    print("✓ No missing values detected!")

# Check for duplicate rows
print("\n" + "=" * 80)
print("DUPLICATE ROWS CHECK")
print("=" * 80)
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")
if duplicate_count > 0:
    print(f"Removing {duplicate_count} duplicate rows...")
    df = df.drop_duplicates()
    print(f"✓ Duplicates removed! New shape: {df.shape}")
else:
    print("✓ No duplicates found!")

# Convert 'Date' column to datetime format
print("\n" + "=" * 80)
print("DATA TYPE CONVERSION")
print("=" * 80)
print("Converting 'Date' column to datetime format...")
df['Date'] = pd.to_datetime(df['Date'])
print("✓ Date column converted to datetime!")

# Ensure numerical columns have correct data types
# Columns that should be numeric: Age, Quantity, Price per Unit, Total Amount
numeric_cols = ['Age', 'Quantity', 'Price per Unit', 'Total Amount']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("✓ Numerical columns verified!")
print("\nData types after cleaning:")
print(df.dtypes)

MISSING VALUES CHECK
✓ No missing values detected!

DUPLICATE ROWS CHECK
Number of duplicate rows: 0
✓ No duplicates found!

DATA TYPE CONVERSION
Converting 'Date' column to datetime format...
✓ Date column converted to datetime!
✓ Numerical columns verified!

Data types after cleaning:
Transaction ID               int64
Date                datetime64[us]
Customer ID                    str
Gender                         str
Age                          int64
Product Category               str
Quantity                     int64
Price per Unit               int64
Total Amount                 int64
dtype: object


## Step 5: Basic Data Validation

Verify data integrity by checking if calculated fields match expected values.

In [5]:
# Validate Total Amount calculation: Total Amount should equal Quantity * Price per Unit
print("=" * 80)
print("TOTAL AMOUNT VALIDATION")
print("=" * 80)

# Calculate expected total amount
df['Expected_Total'] = df['Quantity'] * df['Price per Unit']

# Identify mismatches (with small tolerance for floating point errors)
mismatch_tolerance = 0.01  # Allow 1 cent tolerance for floating point errors
mismatches = df[abs(df['Total Amount'] - df['Expected_Total']) > mismatch_tolerance]

if len(mismatches) > 0:
    print(f"Found {len(mismatches)} mismatches in Total Amount!")
    print("\nSample mismatches:")
    print(mismatches[['Quantity', 'Price per Unit', 'Total Amount', 'Expected_Total']].head())
    
    # Fix the mismatches
    print(f"\nFixing {len(mismatches)} Total Amount values...")
    df.loc[df.index.isin(mismatches.index), 'Total Amount'] = df.loc[df.index.isin(mismatches.index), 'Expected_Total']
    print("✓ Total Amount values corrected!")
else:
    print("✓ All Total Amount values are correct!")

# Drop the temporary Expected_Total column
df = df.drop('Expected_Total', axis=1)

print(f"\nDataset shape after validation: {df.shape}")

TOTAL AMOUNT VALIDATION
✓ All Total Amount values are correct!

Dataset shape after validation: (1000, 9)


## Step 6: Feature Engineering

Create new features from existing data to enhance analysis capabilities.

In [6]:
# Create 'Month' feature from Date column
print("=" * 80)
print("FEATURE ENGINEERING")
print("=" * 80)

df['Month'] = df['Date'].dt.month
print("✓ Created 'Month' column from Date")

# Create 'DayOfWeek' feature from Date column
# 0 = Monday, 6 = Sunday
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['DayOfWeek_Name'] = df['Date'].dt.day_name()
print("✓ Created 'DayOfWeek' column from Date")

# Display sample of new features
print("\nSample of engineered features:")
print(df[['Date', 'Month', 'DayOfWeek', 'DayOfWeek_Name']].head(10))

print(f"\n✓ Feature engineering complete!")
print(f"Updated dataset shape: {df.shape}")

FEATURE ENGINEERING
✓ Created 'Month' column from Date
✓ Created 'DayOfWeek' column from Date

Sample of engineered features:
        Date  Month  DayOfWeek DayOfWeek_Name
0 2023-11-24     11          4         Friday
1 2023-02-27      2          0         Monday
2 2023-01-13      1          4         Friday
3 2023-05-21      5          6         Sunday
4 2023-05-06      5          5       Saturday
5 2023-04-25      4          1        Tuesday
6 2023-03-13      3          0         Monday
7 2023-02-22      2          2      Wednesday
8 2023-12-13     12          2      Wednesday
9 2023-10-07     10          5       Saturday

✓ Feature engineering complete!
Updated dataset shape: (1000, 12)


## Step 7: Key Insights

Extract and display important insights about the dataset.

In [7]:
# Print key insights about the dataset
print("=" * 80)
print("KEY INSIGHTS")
print("=" * 80)

# Number of unique customers
unique_customers = df['Customer ID'].nunique()
print(f"\n1. Number of Unique Customers: {unique_customers}")

# Number of transactions
num_transactions = len(df)
print(f"2. Number of Transactions: {num_transactions}")

# Unique product categories
unique_categories = df['Product Category'].nunique()
product_categories = df['Product Category'].unique()
print(f"3. Unique Product Categories: {unique_categories}")
print(f"   Categories: {', '.join(product_categories)}")

# Additional insights
print(f"\n4. Date Range:")
print(f"   From: {df['Date'].min().date()}")
print(f"   To: {df['Date'].max().date()}")

print(f"\n5. Sales Statistics:")
print(f"   Total Sales Revenue: ${df['Total Amount'].sum():,.2f}")
print(f"   Average Transaction Value: ${df['Total Amount'].mean():,.2f}")
print(f"   Minimum Transaction: ${df['Total Amount'].min():,.2f}")
print(f"   Maximum Transaction: ${df['Total Amount'].max():,.2f}")

print(f"\n6. Customer Demographics:")
print(f"   Average Customer Age: {df['Age'].mean():.1f} years")
print(f"   Age Range: {df['Age'].min():.0f} - {df['Age'].max():.0f} years")
print(f"   Gender Distribution:")
print(df['Gender'].value_counts())

print("\n" + "=" * 80)

KEY INSIGHTS

1. Number of Unique Customers: 1000
2. Number of Transactions: 1000
3. Unique Product Categories: 3
   Categories: Beauty, Clothing, Electronics

4. Date Range:
   From: 2023-01-01
   To: 2024-01-01

5. Sales Statistics:
   Total Sales Revenue: $456,000.00
   Average Transaction Value: $456.00
   Minimum Transaction: $25.00
   Maximum Transaction: $2,000.00

6. Customer Demographics:
   Average Customer Age: 41.4 years
   Age Range: 18 - 64 years
   Gender Distribution:
Gender
Female    510
Male      490
Name: count, dtype: int64



## Conclusion

### Summary

This notebook successfully completed a comprehensive data preprocessing workflow for the retail sales dataset:

**Data Cleaning:**
- ✓ Verified and handled missing values
- ✓ Removed duplicate records
- ✓ Converted temporal data to proper datetime format
- ✓ Ensured numerical columns have appropriate data types

**Data Validation:**
- ✓ Validated Total Amount calculations
- ✓ Corrected any discrepancies found

**Feature Engineering:**
- ✓ Extracted Month from transaction dates
- ✓ Extracted Day of Week information
- ✓ Created categorical features for better analysis

**Data Insights:**
The cleaned dataset now contains comprehensive information about customer transactions, including:
- Customer demographics (gender, age)
- Product categories and quantities purchased
- Sales amounts and pricing
- Temporal information (dates, months, days of week)

